# Required Imports

In [1]:
import os
import re
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image

In [2]:
# Configuration Settings
n_packets = 200
stride = 50
output_folder = './output/'
save_folder = './output_200p/'

# Dataset Generation

## Load IQR Amplitude values 

In [3]:
def extract_id_from_filename(filename):
    match = re.search(r'(\d+[mf][AB])\.', filename)
    return match.group(1) if match else None

def load_feature_dicts(directory):
    feature_dict = {}
    for file_name in os.listdir(directory):
        if file_name.endswith('.csv'):
            df = pd.read_csv(os.path.join(directory, file_name))
            file_id = extract_id_from_filename(file_name)
            if file_id:
                feature_dict[file_id] = df
    return feature_dict

def save_heatmap_image(data, output_path):
    plt.figure()
    plt.imshow(data, aspect='auto')
    plt.axis('off')
    buffer = BytesIO()
    plt.savefig(buffer, format='png', bbox_inches='tight', pad_inches=0)
    plt.close()
    buffer.seek(0)
    Image.open(buffer).convert('RGB').save(output_path, format='PNG')

# 1. Load both sanitized amplitudes and raw phases
amp_dir = os.path.join(output_folder, 'CSV', 'sanitized_amplitudes', 'IQR_amplitudes')
phase_dir = os.path.join(output_folder, 'CSV', 'raw_phases')
sanitized_amplitudes = load_feature_dicts(amp_dir)
sanitized_phases = load_feature_dicts(phase_dir)

# 2. Slice into 200-packet sliding windows with stride=50
split_data = {}
for ID in sanitized_amplitudes.keys():
    split_data[ID] = []
    amp_df = sanitized_amplitudes[ID]
    phase_df = sanitized_phases[ID]
    total_len = min(len(amp_df), len(phase_df))

    for s in range(0, total_len - n_packets + 1, stride):
        e = s + n_packets
        amp_mat = amp_df.iloc[s:e].drop(columns='timestamp').to_numpy()
        phase_mat = phase_df.iloc[s:e].drop(columns='timestamp').to_numpy()
        split_data[ID].append((amp_mat, phase_mat))

# 3. Split 40 Train / 41 Test identities with seed=42
random.seed(42)
groups = {}
for key, packets in split_data.items():
    base_id, session = key[:-1], key[-1]
    groups.setdefault(base_id, {'A': [], 'B': []})[session].extend(packets)

all_identities = list(groups.keys())
random.shuffle(all_identities)
train_ids, test_ids = all_identities[:40], all_identities[40:]

splits = {
    ('D1', 'Probe'):   {i: (groups[i]['A'] + groups[i]['B'])[:len(groups[i]['A'] + groups[i]['B']) // 2] for i in train_ids},
    ('D1', 'Gallery'): {i: (groups[i]['A'] + groups[i]['B'])[len(groups[i]['A'] + groups[i]['B']) // 2:] for i in train_ids},
    ('D2t', 'Probe'):  {i: groups[i]['B'] for i in test_ids},
    ('D2t', 'Gallery'):{i: groups[i]['A'] for i in test_ids},
}

# 4. Save paired _amp.png and _phase.png heatmaps
for (subset, role), data_dict in splits.items():
    out_dir = os.path.join(save_folder, 'ImageData_AmpPhase', subset, role)
    os.makedirs(out_dir, exist_ok=True)
    for ID, pair_list in data_dict.items():
        for idx, (amp_mat, phase_mat) in enumerate(pair_list):
            save_heatmap_image(amp_mat, os.path.join(out_dir, f"{ID}_{idx}_amp.png"))
            save_heatmap_image(phase_mat, os.path.join(out_dir, f"{ID}_{idx}_phase.png"))

print(f"Done! Saved paired Amplitude & Phase heatmaps to {os.path.join(save_folder, 'ImageData_AmpPhase/')}")

Done! Saved paired Amplitude & Phase heatmaps to ./output_200p/ImageData_AmpPhase/


In [4]:
import os, re, random, torch
import pandas as pd
import numpy as np

n_packets = 200
stride = 50
output_folder = './output/'
save_folder = './output_200p/'

def load_csvs(directory):
    d = {}
    for f in os.listdir(directory):
        if f.endswith('.csv'):
            m = re.search(r'(\d+[mf][AB])\.', f)
            if m:
                d[m.group(1)] = pd.read_csv(os.path.join(directory, f))
    return d

amps = load_csvs(os.path.join(output_folder, 'CSV', 'sanitized_amplitudes', 'IQR_amplitudes'))
phases = load_csvs(os.path.join(output_folder, 'CSV', 'raw_phases'))

groups = {}
for key in amps.keys():
    base_id, session = key[:-1], key[-1]
    groups.setdefault(base_id, {'A': [], 'B': []})
    total_len = min(len(amps[key]), len(phases[key]))
    for s in range(0, total_len - n_packets + 1, stride):
        e = s + n_packets
        a = amps[key].iloc[s:e].drop(columns='timestamp').to_numpy(dtype=np.float32)   # [200, 52]
        p = phases[key].iloc[s:e].drop(columns='timestamp').to_numpy(dtype=np.float32) # [200, 52]
        tensor_2ch = torch.from_numpy(np.stack([a, p], axis=0))                        # [2, 200, 52]
        groups[base_id][session].append(tensor_2ch)

random.seed(42)
all_ids = list(groups.keys())
random.shuffle(all_ids)
train_ids, test_ids = all_ids[:40], all_ids[40:]

splits = {
    ('D1', 'Probe'):    {i: (groups[i]['A'] + groups[i]['B'])[:len(groups[i]['A'] + groups[i]['B']) // 2] for i in train_ids},
    ('D1', 'Gallery'):  {i: (groups[i]['A'] + groups[i]['B'])[len(groups[i]['A'] + groups[i]['B']) // 2:] for i in train_ids},
    ('D2t', 'Probe'):   {i: groups[i]['B'] for i in test_ids},
    ('D2t', 'Gallery'): {i: groups[i]['A'] for i in test_ids},
}

for (subset, role), data_dict in splits.items():
    out_dir = os.path.join(save_folder, 'RawTensorData_AmpPhase', subset, role)
    os.makedirs(out_dir, exist_ok=True)
    for ID, tensors in data_dict.items():
        for idx, t in enumerate(tensors):
            torch.save(t, os.path.join(out_dir, f"{ID}_{idx}.pt"))

print(f"Done! Saved raw [2, {n_packets}, 52] CNN tensors to {os.path.join(save_folder, 'RawTensorData_AmpPhase/')}")

Done! Saved raw [2, 200, 52] CNN tensors to ./output_200p/RawTensorData_AmpPhase/
